# 예산 가드와 대화당 비용 p95

개별 호출이 아니라 대화 단위로 비용을 합치고 p95가 목표치를 넘는지 판정합니다.

## 실행 환경 준비

## 코드 읽기와 개념 경계

대화당 비용은 개별 호출 행을 `conversation` 식별자로 묶어 합한 값입니다. `nearest_rank`는 값을 정렬하고 `ceil(percentile × n)`번째 값을 고릅니다. 표본이 열 개이면 p95가 사실상 가장 큰 관찰값이 되므로 출력의 p95는 한 대화에 민감합니다. 계산 도구마다 보간 방식이 다를 수 있어 지표 이름과 방식을 함께 기록합니다.

`allow_call(spent, estimated_max, hard_limit)`은 이미 쓴 값과 다음 호출의 예상 최대값을 더해 요청 전에 실행 여부를 반환합니다. 실제 사용량을 사후 집계하는 p95와 호출 전 차단하는 예산 가드는 시점이 다릅니다. 예상치를 너무 낮게 두면 상한을 넘을 수 있고 너무 높게 두면 정상 요청을 막으므로 모델·도구 경로별 상한을 관찰 자료로 갱신합니다.

In [1]:
import math
print("비용 집계 준비 완료")

비용 집계 준비 완료


## 대화별 호출 기록

각 대화는 한 번 이상의 모델 호출을 포함합니다. 아래 값은 비용 계산을 학습하기 위한 고정 기록입니다.

In [2]:
calls = [
 {"conversation":"c1","cost":0.0012}, {"conversation":"c1","cost":0.0008},
 {"conversation":"c2","cost":0.0021}, {"conversation":"c3","cost":0.0017},
 {"conversation":"c4","cost":0.0062}, {"conversation":"c5","cost":0.0014},
 {"conversation":"c6","cost":0.0025}, {"conversation":"c7","cost":0.0018},
 {"conversation":"c8","cost":0.0030}, {"conversation":"c9","cost":0.0016},
 {"conversation":"c10","cost":0.0048}]
totals = {}
for row in calls: totals[row["conversation"]] = totals.get(row["conversation"], 0) + row["cost"]
print(totals)

{'c1': 0.002, 'c2': 0.0021, 'c3': 0.0017, 'c4': 0.0062, 'c5': 0.0014, 'c6': 0.0025, 'c7': 0.0018, 'c8': 0.003, 'c9': 0.0016, 'c10': 0.0048}


## 최근접 순위 방식 p95

값을 정렬한 뒤 `ceil(0.95 × n)`번째 값을 고릅니다. 계산 방식을 대시보드 문서에 함께 기록해야 합니다.

In [3]:
def nearest_rank(values, percentile):
    ordered = sorted(values)
    rank = math.ceil(percentile * len(ordered))
    return ordered[rank - 1]

costs = list(totals.values())
p50 = nearest_rank(costs, 0.50)
p95 = nearest_rank(costs, 0.95)
print(f"대화당 비용 p50=${p50:.4f}, p95=${p95:.4f}")

대화당 비용 p50=$0.0020, p95=$0.0062


## 예산 가드 판정

요청 전에 예상 최대 비용을 더해 상한을 넘으면 호출을 시작하지 않습니다.

In [4]:
def allow_call(spent, estimated_max, hard_limit):
    return spent + estimated_max <= hard_limit

checks = [(0.003, 0.001, 0.005), (0.0045, 0.001, 0.005)]
for values in checks:
    print(values, "허용" if allow_call(*values) else "차단")

(0.003, 0.001, 0.005) 허용
(0.0045, 0.001, 0.005) 차단


## 목표치와 비교

목표를 0.005달러로 두고 p95를 판정합니다. 표본이 10개라서 한 개의 큰 값에 민감합니다.

In [5]:
target = 0.005
print("목표:", target, "관찰 p95:", p95, "상태:", "초과" if p95 > target else "충족")
assert p95 == max(costs)

목표: 0.005 관찰 p95: 0.0062 상태: 초과
